In [1]:
import json, os

os.makedirs('/workspace/llm_project/data', exist_ok=True)

vision_qa = [
  # ===== COATING 공정 =====
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "현재 코팅(Coating) 공정이 진행 중입니다. 슬롯 다이(Slot Die)를 통해 전극 포일 위에 활물질 슬러리가 균일하게 도포되고 있습니다. 코팅 두께와 균일성이 배터리 용량에 직접 영향을 미치므로 정밀한 제어가 필요합니다."},
  {"instruction": "지금 보이는 장비는 뭐야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "현재 화면에는 슬롯 다이(Slot Die)와 전극 포일(Electrode Foil)이 보입니다. 슬롯 다이는 슬러리를 균일하게 도포하는 핵심 장비이며, 전극 포일은 활물질이 도포되는 집전체입니다."},
  {"instruction": "이 공정에서 주의할 점은?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정에서 주의할 점은 첫째, 슬롯 다이 갭을 일정하게 유지하여 코팅 두께 균일성을 확보해야 합니다. 둘째, 슬러리 점도와 코팅 속도를 최적화해야 합니다. 셋째, 핀홀, 줄무늬, 비딩 등 표면 결함을 실시간으로 모니터링해야 합니다."},
  {"instruction": "이 공정 다음엔 뭐가 진행돼?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 이후에는 캘린더링(Calendering) 공정이 진행됩니다. 코팅된 전극을 고압 롤러로 압연하여 활물질 밀도를 높이고 두께를 균일하게 만드는 공정입니다."},
  {"instruction": "슬롯 다이가 뭐야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "슬롯 다이(Slot Die)는 코팅 공정의 핵심 장비로, 슬러리를 균일한 두께로 전극 포일에 도포하는 노즐입니다. 다이 갭(슬롯 간격)이 코팅 두께를 결정하며, 마이크로미터 수준의 정밀도로 제어됩니다."},
  {"instruction": "코팅이 불균일하면 어떻게 돼?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅이 불균일하면 전극의 활물질 분포가 고르지 않아 배터리 용량이 저하됩니다. 또한 충방전 시 국부적인 과부하가 발생하여 배터리 수명이 단축되고 최악의 경우 내부 단락으로 이어질 수 있습니다."},
  {"instruction": "지금 공정 정상이야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "슬롯 다이와 전극 포일이 정상적으로 탐지되었습니다. 코팅 공정이 진행 중입니다. 코팅 두께 균일성과 슬러리 도포 상태를 지속적으로 모니터링하여 핀홀, 줄무늬 등 표면 결함 여부를 확인하세요."},

  # ===== CALENDERING 공정 =====
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "현재 캘린더링(Calendering) 공정이 진행 중입니다. 고압 프레스 롤러(Press Roller)가 코팅된 전극 포일을 압연하여 활물질 밀도를 높이고 두께를 균일하게 만들고 있습니다."},
  {"instruction": "지금 보이는 장비는 뭐야?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "현재 화면에는 프레스 롤러(Press Roller)와 전극 포일(Electrode Foil)이 보입니다. 프레스 롤러는 수십에서 수백 MPa의 고압으로 전극을 압연하는 장비입니다."},
  {"instruction": "이 공정에서 주의할 점은?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "캘린더링 공정에서 주의할 점은 첫째, 롤러 압력을 정밀하게 제어하여 목표 두께를 달성해야 합니다. 둘째, 롤러 온도를 일정하게 유지해야 합니다. 셋째, 스프링백 현상을 고려하여 압연 압력을 설정해야 합니다."},
  {"instruction": "이 공정 다음엔 뭐가 진행돼?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "캘린더링 공정 이후에는 슬리팅(Slitting) 공정이 진행됩니다. 압연된 광폭 전극을 원형 커터로 설계된 폭으로 정밀하게 절단하는 공정입니다."},
  {"instruction": "압연 압력이 너무 높으면 어떻게 돼?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "압연 압력이 너무 높으면 전극이 과도하게 압축되어 공극률이 낮아집니다. 공극률이 너무 낮으면 전해액 침투가 어려워져 이온 전도성이 저하되고 출력 특성이 나빠집니다. 또한 집전체 포일이 찢어지는 불량이 발생할 수 있습니다."},
  {"instruction": "지금 공정 정상이야?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "프레스 롤러와 전극 포일이 정상적으로 탐지되었습니다. 캘린더링 공정이 진행 중입니다. 롤러 압력과 전극 두께를 실시간으로 모니터링하고 스프링백 현상에 의한 두께 복원을 고려하여 공정 조건을 유지하세요."},

  # ===== SLITTING 공정 =====
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "현재 슬리팅(Slitting) 공정이 진행 중입니다. 슬리팅 칼날(Slitting Blade)이 광폭 전극 포일을 설계된 폭으로 정밀하게 절단하고 있습니다."},
  {"instruction": "지금 보이는 장비는 뭐야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "현재 화면에는 슬리팅 블레이드(Slitting Blade)와 전극 포일(Electrode Foil)이 보입니다. 슬리팅 블레이드는 원형 커터로 전극을 정밀하게 절단하는 장비입니다."},
  {"instruction": "이 공정에서 주의할 점은?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정에서 주의할 점은 첫째, 커터 마모 상태를 주기적으로 점검하여 버(Burr) 발생을 방지해야 합니다. 둘째, 집진 시스템을 가동하여 금속 분말이 전극에 혼입되지 않도록 해야 합니다. 셋째, 절단면 품질을 실시간으로 검사해야 합니다."},
  {"instruction": "이 공정 다음엔 뭐가 진행돼?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정 이후에는 와인딩(Winding) 공정이 진행됩니다. 절단된 전극과 분리막을 맨드릴에 권취하여 젤리롤 형태의 전극 조립체를 만드는 공정입니다."},
  {"instruction": "버가 발생하면 어떻게 해야 해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "버(Burr)가 발생하면 즉시 라인을 정지하고 슬리팅 블레이드의 마모 상태를 확인해야 합니다. 버가 발생한 전극 구간을 마킹하여 와인딩에 사용하지 않도록 격리합니다. 블레이드 교체 후 절단면 품질을 재확인하고 생산을 재개합니다."},
  {"instruction": "지금 공정 정상이야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 블레이드와 전극 포일이 정상적으로 탐지되었습니다. 슬리팅 공정이 진행 중입니다. 절단면에 버 발생 여부와 금속 분말 집진 상태를 확인하고, 커터 마모 상태를 주기적으로 점검하세요."},
  {"instruction": "슬리팅 후 전극 폭이 불균일하면?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 후 전극 폭이 불균일하면 와인딩 시 전극 정렬 불량이 발생합니다. 이는 젤리롤 내부에서 양극과 음극의 오버행 설계가 무너져 내부 단락 위험이 증가합니다. 커터 위치와 장력 제어 시스템을 점검해야 합니다."},

  # ===== WINDING 공정 =====
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "현재 와인딩(Winding) 공정이 진행 중입니다. 와인더(Winder)가 전극 포일과 분리막(Separator)을 맨드릴에 권취하여 젤리롤 형태의 전극 조립체를 만들고 있습니다."},
  {"instruction": "지금 보이는 장비는 뭐야?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "현재 화면에는 와인더(Winder), 분리막(Separator), 전극 포일(Electrode Foil)이 보입니다. 와인더는 전극과 분리막을 정밀하게 권취하는 장비이며, 분리막은 양극과 음극의 직접 접촉을 방지하는 핵심 소재입니다."},
  {"instruction": "이 공정에서 주의할 점은?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정에서 주의할 점은 첫째, 전극과 분리막의 정렬 상태를 실시간으로 모니터링해야 합니다. 둘째, 권취 장력을 균일하게 유지하여 공기 포켓 발생을 방지해야 합니다. 셋째, 분리막 손상 여부를 지속적으로 확인해야 합니다."},
  {"instruction": "이 공정 다음엔 뭐가 진행돼?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정 이후에는 탭 용접(Tab Welding)과 케이스 조립 공정이 진행됩니다. 완성된 젤리롤에 탭을 용접하고 원통형 케이스에 삽입한 후 전해액을 주입합니다."},
  {"instruction": "분리막이 뭐야?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "분리막(Separator)은 폴리에틸렌(PE) 또는 폴리프로필렌(PP) 소재의 다공성 필름입니다. 양극과 음극의 직접 접촉을 막아 내부 단락을 방지하면서 리튬 이온이 통과할 수 있는 미세 기공을 가지고 있습니다."},
  {"instruction": "지금 공정 정상이야?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인더, 분리막, 전극 포일이 정상적으로 탐지되었습니다. 와인딩 공정이 진행 중입니다. 전극과 분리막의 정렬 상태와 권취 장력을 확인하고, 젤리롤 직경이 설계값 범위 내에 있는지 모니터링하세요."},
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Winding_Cell, Numbered_Core", "output": "현재 와인딩(Winding) 공정이 진행 중입니다. 번호가 매겨진 코어(Numbered Core)에 전극이 권취되어 와인딩 셀(Winding Cell)이 형성되고 있습니다. 각 코어 번호는 롤러 위치를 식별하여 유지보수 시 신속한 점검을 가능하게 합니다."},
  {"instruction": "지금 보이는 장비는 뭐야?", "input": "감지된 객체: Winding_Cell, Numbered_Core", "output": "현재 화면에는 와인딩 셀(Winding Cell)과 번호가 매겨진 코어(Numbered Core)가 보입니다. 번호 코어는 와인딩 장비 내부의 가이드 롤러로, 번호를 통해 각 롤러의 위치와 역할을 명확히 파악할 수 있습니다."},
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Separator, Numbered_Core, Electrode_Foil", "output": "현재 와인딩(Winding) 공정이 진행 중입니다. 분리막(Separator)과 전극 포일(Electrode Foil)이 번호 코어(Numbered Core) 롤러들을 통해 이송되며 권취되고 있습니다. 분리막과 전극의 장력 균형이 젤리롤 품질을 결정합니다."},
  {"instruction": "Core_Shaft가 뭐야?", "input": "감지된 객체: Core_Shaft, Winder", "output": "Core Shaft(코어 샤프트)는 와인딩 공정에서 젤리롤을 권취하는 맨드릴의 중심축입니다. 전극과 분리막이 이 축을 중심으로 감겨 원통형 젤리롤이 형성됩니다. 권취 완료 후 코어 샤프트가 빠져나오면 젤리롤 중심에 구멍이 남습니다."},
  {"instruction": "지금 여기선 뭘 하고 있는거야?", "input": "감지된 객체: Core_Shaft, Winder, Separator", "output": "현재 와인딩(Winding) 공정이 진행 중입니다. 코어 샤프트(Core Shaft)를 중심으로 분리막(Separator)과 전극이 와인더(Winder)에 의해 권취되고 있습니다. 이 공정을 통해 원통형 젤리롤이 완성됩니다."},

  # ===== 복합 질문 =====
  {"instruction": "지금 어느 공정 단계야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "현재 배터리 전극 제조 4단계 중 1단계인 코팅(Coating) 공정입니다. 슬롯 다이를 통해 전극 포일에 활물질을 도포하는 단계로, 이후 캘린더링 → 슬리팅 → 와인딩 순서로 진행됩니다."},
  {"instruction": "지금 어느 공정 단계야?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "현재 배터리 전극 제조 4단계 중 2단계인 캘린더링(Calendering) 공정입니다. 코팅이 완료된 전극을 압연하는 단계로, 이후 슬리팅 → 와인딩 순서로 진행됩니다."},
  {"instruction": "지금 어느 공정 단계야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "현재 배터리 전극 제조 4단계 중 3단계인 슬리팅(Slitting) 공정입니다. 압연된 전극을 설계 폭으로 절단하는 단계로, 이후 마지막 공정인 와인딩이 진행됩니다."},
  {"instruction": "지금 어느 공정 단계야?", "input": "감지된 객체: Winder, Separator", "output": "현재 배터리 전극 제조 4단계 중 마지막 4단계인 와인딩(Winding) 공정입니다. 전극과 분리막을 권취하여 젤리롤을 완성하는 최종 단계입니다."},
  {"instruction": "이 장비에서 불량이 발생하면 다음 공정에 어떤 영향을 줘?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정에서 버나 폭 불균일 불량이 발생하면 와인딩 공정에서 전극 정렬 불량이 발생합니다. 정렬 불량은 젤리롤 내부에서 양극과 음극이 오버랩되어 내부 단락을 유발하며, 이는 배터리 발화 위험으로 이어질 수 있습니다."},
  {"instruction": "이 장비에서 불량이 발생하면 다음 공정에 어떤 영향을 줘?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정에서 두께 불균일이나 핀홀 불량이 발생하면 캘린더링 시 압연 편차가 발생하고 슬리팅 시 절단면 불량으로 이어집니다. 최종적으로 와인딩에서 젤리롤 내부 밀도 불균일로 배터리 성능이 저하됩니다."},
]

with open('/workspace/llm_project/data/vision_qa.json', 'w', encoding='utf-8') as f:
    json.dump(vision_qa, f, ensure_ascii=False, indent=2)

print(f"저장 완료! 총 {len(vision_qa)}개")

저장 완료! 총 37개


In [2]:
import json

with open('/workspace/llm_project/data/vision_qa.json', 'r', encoding='utf-8') as f:
    vision_qa = json.load(f)

additional = [
  # ===== COATING 추가 =====
  {"instruction": "지금 공정에서 어떤 불량이 발생할 수 있어?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정에서 발생할 수 있는 주요 불량은 핀홀(Pinhole), 줄무늬(Streak), 비딩(Beading), 두께 불균일입니다. 핀홀은 슬러리 내 기포나 이물질로 발생하고, 줄무늬는 슬롯 다이 막힘으로 발생합니다. 이러한 불량은 배터리 용량 저하와 내부 단락 위험을 유발합니다."},
  {"instruction": "전극 포일이 뭐야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "전극 포일(Electrode Foil)은 배터리 전극의 집전체입니다. 양극은 알루미늄 포일, 음극은 구리 포일을 사용합니다. 포일 위에 활물질 슬러리를 코팅하여 전극을 만들며, 포일의 두께와 표면 상태가 코팅 품질에 영향을 미칩니다."},
  {"instruction": "코팅 속도가 너무 빠르면 어떻게 돼?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 속도가 너무 빠르면 슬러리가 포일에 충분히 도포되지 않아 코팅 두께가 얇아지고 핀홀이 발생하기 쉽습니다. 반대로 너무 느리면 슬러리가 흘러내려 두께 불균일이 발생합니다. 코팅 속도는 슬러리 점도와 목표 두께에 맞게 최적화해야 합니다."},
  {"instruction": "슬러리가 뭐야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "슬러리(Slurry)는 활물질, 도전재, 바인더를 용매에 혼합한 페이스트 형태의 재료입니다. 양극 슬러리는 LiCoO2 등의 양극재를, 음극 슬러리는 흑연 등의 음극재를 포함합니다. 슬러리의 점도와 고형분 비율이 코팅 품질을 결정합니다."},
  {"instruction": "코팅 후 건조는 왜 필요해?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 후 건조는 슬러리에 포함된 용매(NMP 또는 물)를 제거하기 위해 필요합니다. 용매가 남아있으면 캘린더링 시 전극이 변형되고 배터리 내부에서 가스가 발생합니다. 건조로에서 온도와 시간을 제어하여 완전히 건조시킵니다."},

  # ===== CALENDERING 추가 =====
  {"instruction": "지금 공정에서 어떤 불량이 발생할 수 있어?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "캘린더링 공정에서 발생할 수 있는 주요 불량은 전극 크랙, 주름, 두께 불균일, 활물질 탈락입니다. 압연 압력이 너무 높으면 크랙이 발생하고, 장력이 불균일하면 주름이 생깁니다. 롤러 표면 이물질은 전극 표면에 압흔을 남깁니다."},
  {"instruction": "스프링백 현상이 뭐야?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "스프링백(Spring-back) 현상은 캘린더링 후 전극이 롤러 압력에서 벗어날 때 탄성 복원력으로 인해 두께가 일부 되돌아오는 현상입니다. 목표 두께를 달성하려면 스프링백 양을 예측하여 압연 압력을 더 높게 설정해야 합니다."},
  {"instruction": "캘린더링 후 전극 밀도가 올라가면 어떤 장점이 있어?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "캘린더링으로 전극 밀도가 높아지면 단위 부피당 에너지 밀도가 증가합니다. 또한 활물질과 집전체 포일 간의 접촉 저항이 낮아져 전기 전도성이 향상됩니다. 전극 두께가 균일해져 이후 슬리팅과 와인딩 공정의 품질도 향상됩니다."},
  {"instruction": "롤러 온도를 왜 관리해야 해?", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "롤러 온도가 너무 높으면 바인더가 연화되어 활물질이 롤러 표면에 달라붙는 문제가 발생합니다. 온도가 낮으면 압연 효율이 떨어집니다. 일부 장비는 롤러 내부에 냉각수 또는 가열 매체를 순환시켜 롤러 온도를 일정하게 유지합니다."},

  # ===== SLITTING 추가 =====
  {"instruction": "지금 공정에서 어떤 불량이 발생할 수 있어?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정에서 발생할 수 있는 주요 불량은 버(Burr), 폭 불균일, 사행(蛇行), 텔레스코핑입니다. 버는 커터 마모로 발생하고, 사행은 장력 불균일로 전극이 좌우로 흔들리는 현상입니다. 텔레스코핑은 롤 단면이 어긋나 계단 형태가 되는 불량입니다."},
  {"instruction": "슬리팅 블레이드는 얼마나 자주 교체해야 해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 블레이드 교체 주기는 전극 소재와 생산량에 따라 다르지만 일반적으로 수십만 미터 절단 후 교체합니다. 절단면 품질 검사에서 버 발생이 증가하면 즉시 교체해야 합니다. 예방 차원에서 정기 점검을 통해 마모 상태를 확인합니다."},
  {"instruction": "금속 분말이 전극에 혼입되면 왜 위험해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 시 발생하는 구리 또는 알루미늄 금속 분말이 전극에 혼입되면 와인딩 후 젤리롤 내부에서 미세 단락을 유발합니다. 미세 단락은 충방전 시 발열로 이어지고 최악의 경우 배터리 열폭주와 발화를 초래합니다. 집진 시스템 유지가 필수적입니다."},
  {"instruction": "EPC 시스템이 뭐야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "EPC(Edge Position Control) 시스템은 슬리팅 공정에서 전극의 사행(좌우 흔들림)을 자동으로 보정하는 장치입니다. 센서가 전극 엣지 위치를 실시간으로 감지하고 가이드 롤러 위치를 자동 조절하여 전극이 항상 정확한 위치로 이송되도록 합니다."},

  # ===== WINDING 추가 =====
  {"instruction": "지금 공정에서 어떤 불량이 발생할 수 있어?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정에서 발생할 수 있는 주요 불량은 전극 정렬 불량, 분리막 손상, 공기 포켓, 젤리롤 직경 편차입니다. 전극 정렬 불량은 내부 단락 위험을 유발하고, 분리막 손상은 즉각적인 단락을 초래합니다. 공기 포켓은 전해액 함침 불량으로 이어집니다."},
  {"instruction": "오버행이 뭐야?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "오버행(Overhang)은 와인딩 시 분리막이 전극보다 양쪽으로 더 넓게 돌출된 구조입니다. 분리막 오버행은 전극 정렬 오차가 생겨도 양극과 음극이 직접 접촉하지 않도록 안전 마진을 제공합니다. 일반적으로 전극 양쪽으로 0.5~2mm 더 넓게 설계합니다."},
  {"instruction": "젤리롤이 완성되면 다음엔 뭐가 진행돼?", "input": "감지된 객체: Winding_Cell, Numbered_Core", "output": "젤리롤이 완성되면 먼저 단락 시험(Short Circuit Test)으로 내부 불량을 검사합니다. 합격한 젤리롤은 탭 용접(Tab Welding) 공정으로 이동하여 양극과 음극 탭을 용접합니다. 이후 원통형 캔에 삽입하고 전해액을 주입하여 배터리 셀이 완성됩니다."},
  {"instruction": "권취 장력이 불균일하면 어떻게 돼?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "권취 장력이 불균일하면 젤리롤 내부 층간 압력이 고르지 않아 전극과 분리막 사이에 공기 포켓이 생깁니다. 공기 포켓은 전해액 함침을 방해하고 충방전 시 전극 팽창으로 분리막을 손상시킵니다. 균일한 장력 유지가 젤리롤 품질의 핵심입니다."},
  {"instruction": "분리막이 손상되면 어떻게 돼?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "분리막이 손상되면 양극과 음극이 직접 접촉하여 내부 단락이 발생합니다. 내부 단락은 대전류 흐름으로 인한 급격한 발열을 유발하고 최악의 경우 열폭주로 이어져 배터리 발화 및 폭발을 초래합니다. 분리막 손상은 가장 위험한 배터리 불량입니다."},
  {"instruction": "와인딩 속도가 너무 빠르면 어떻게 돼?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 속도가 너무 빠르면 전극과 분리막의 정렬 제어가 어려워져 정렬 불량이 발생합니다. 또한 급격한 장력 변동으로 분리막이 늘어나거나 찢어질 수 있습니다. 고속 와인딩에서는 정밀한 장력 제어와 비전 정렬 시스템이 필수적입니다."},

  # ===== 일반 공정 질문 =====
  {"instruction": "배터리 전극 제조 공정 순서를 설명해줘.", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "배터리 전극 제조는 4단계로 진행됩니다. 1단계 코팅(Coating): 슬롯 다이로 전극 포일에 활물질 슬러리를 도포합니다. 2단계 캘린더링(Calendering): 고압 롤러로 압연하여 밀도를 높입니다. 3단계 슬리팅(Slitting): 원형 커터로 설계 폭으로 절단합니다. 4단계 와인딩(Winding): 전극과 분리막을 권취하여 젤리롤을 완성합니다."},
  {"instruction": "지금 이 공정이 배터리 품질에 얼마나 중요해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정은 배터리 품질에 매우 중요합니다. 절단면의 버 발생은 분리막 손상과 내부 단락으로 이어질 수 있어 안전성에 직결됩니다. 전극 폭 정확도는 와인딩 정렬에 영향을 미쳐 배터리 용량과 수명을 결정합니다. 슬리팅 품질이 곧 배터리 안전성의 기초입니다."},
  {"instruction": "지금 이 공정이 배터리 품질에 얼마나 중요해?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정은 배터리 제조의 최종 조립 단계로 매우 중요합니다. 전극 정렬 정확도가 배터리 용량과 안전성을 결정합니다. 분리막 손상 없이 균일하게 권취된 젤리롤이 고품질 배터리의 기반입니다. 와인딩 불량은 이후 공정에서 수정이 불가능하므로 실시간 품질 관리가 필수입니다."},
  {"instruction": "AI 비전 시스템이 이 공정에서 어떤 역할을 해?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "AI 비전 시스템은 코팅 공정에서 실시간으로 슬롯 다이와 전극 포일을 탐지하여 공정 상태를 모니터링합니다. 코팅 두께 불균일, 핀홀, 줄무늬 등의 결함을 자동으로 감지하고 즉시 알람을 발생시킵니다. 24시간 일관된 품질 기준으로 검사하여 불량품이 다음 공정으로 넘어가는 것을 방지합니다."},
  {"instruction": "AI 비전 시스템이 이 공정에서 어떤 역할을 해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "AI 비전 시스템은 슬리팅 공정에서 슬리팅 블레이드와 전극 포일을 실시간으로 탐지하여 절단면 품질을 모니터링합니다. 버 발생을 조기에 감지하여 불량 전극이 와인딩 공정으로 투입되는 것을 차단합니다. 커터 마모 패턴을 분석하여 예방적 유지보수 시점도 예측할 수 있습니다."},
  {"instruction": "AI 비전 시스템이 이 공정에서 어떤 역할을 해?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "AI 비전 시스템은 와인딩 공정에서 와인더, 분리막, 전극 포일을 실시간으로 탐지하여 권취 상태를 모니터링합니다. 전극 정렬 편차를 실시간으로 감지하고 자동 보정 신호를 전송합니다. 젤리롤 직경 편차와 분리막 손상 여부를 즉시 파악하여 불량 젤리롤을 자동으로 격리합니다."},
]

vision_qa.extend(additional)

with open('/workspace/llm_project/data/vision_qa.json', 'w', encoding='utf-8') as f:
    json.dump(vision_qa, f, ensure_ascii=False, indent=2)

print(f"저장 완료! 총 {len(vision_qa)}개")

저장 완료! 총 62개


In [3]:
import json

with open('/workspace/llm_project/data/vision_qa.json', 'r', encoding='utf-8') as f:
    vision_qa = json.load(f)

additional2 = [
  # ===== 영상 장면 기반 상세 설명 =====
  # 코팅 구간 - 슬롯 다이 클로즈업
  {"instruction": "슬롯 다이에서 슬러리가 나오는 모습이 보여. 지금 뭐가 일어나고 있는거야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "슬롯 다이의 좁은 슬롯을 통해 활물질 슬러리가 균일하게 압출되어 전극 포일 위에 도포되고 있습니다. 슬롯 다이와 포일 사이의 간격(코팅 갭)이 코팅 두께를 결정합니다. 슬러리가 끊김 없이 일정하게 흘러나오는 것이 정상 상태이며, 맥동이나 줄무늬가 보이면 즉시 점검이 필요합니다."},
  {"instruction": "구리색 포일이 롤러 사이를 통과하고 있어. 이게 뭐야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "구리색 포일은 음극 집전체인 구리 포일(Copper Foil)입니다. 현재 코팅 장비의 백업 롤러를 통과하며 슬롯 다이 아래를 지나고 있습니다. 포일이 처지거나 흔들리지 않도록 백업 롤러가 받쳐주고 있으며, 이 구간에서 음극 활물질 슬러리가 도포됩니다."},
  {"instruction": "코팅 장비 옆에 산업용 카메라가 보여. 저 카메라는 뭘 하는거야?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 인라인 비전 검사 카메라입니다. 건조로를 통과한 직후 전극 표면을 실시간으로 촬영하여 핀홀, 줄무늬, 코팅 누락 등의 표면 결함을 자동으로 탐지합니다. 결함이 감지되면 즉시 알람이 발생하고 해당 구간이 마킹되어 불량 전극이 다음 공정으로 투입되는 것을 방지합니다."},
  {"instruction": "코팅된 전극이 대형 건조로로 들어가고 있어. 건조로 온도는 얼마나 돼?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "배터리 전극 건조로 온도는 일반적으로 100~150°C 범위입니다. NMP 용매를 사용하는 양극 코팅은 130°C 이상, 수계 슬러리를 사용하는 음극 코팅은 100~120°C 수준입니다. 건조 시간과 온도 프로파일이 용매 제거율과 전극 품질을 결정하므로 정밀한 온도 제어가 필요합니다."},

  # 슬리팅 구간
  {"instruction": "원형 커터 위에 파란색 집진 장치가 보여. 저게 뭐야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "파란색 장치는 집진 흡입 노즐입니다. 슬리팅 시 원형 커터가 전극을 절단할 때 발생하는 구리 또는 알루미늄 금속 분말과 활물질 미분을 즉시 흡입하여 포집합니다. 커터 바로 위에 설치되어 분말 발생 즉시 포집 효율을 최대화합니다. 이 장치가 작동하지 않으면 금속 분말이 전극에 낙하하여 내부 단락 위험이 생깁니다."},
  {"instruction": "슬리팅 후 전극이 여러 가닥으로 나뉘어 각각 롤에 감기고 있어. 이게 정상이야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "정상적인 슬리팅 공정입니다. 광폭 전극이 여러 개의 원형 커터로 동시에 절단되어 각각 별도의 리와인더 코어에 감깁니다. 각 가닥의 폭이 균일하고 롤 단면이 가지런하게 감기면 정상입니다. 롤 단면이 계단처럼 어긋나는 텔레스코핑 현상이 보이면 장력 제어를 점검해야 합니다."},
  {"instruction": "슬리팅 검사 시스템 화면에 흑백 영상과 수치가 표시돼. 저게 뭘 측정하는거야?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 인라인 검사 시스템이 전극 폭과 코팅 경계선을 실시간으로 측정하고 있습니다. 흑백 라인 스캔 카메라가 전극 엣지를 연속 촬영하여 활물질 코팅 폭과 무지부 폭을 계산합니다. 표시된 수치는 현재 측정값과 설정값의 편차를 나타내며, 허용 범위를 초과하면 즉시 알람이 발생합니다."},
  {"instruction": "전극에 검은 활물질 부분과 구리색 무지부가 선명하게 보여. 무지부가 왜 필요해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "무지부(Bare Area)는 활물질이 코팅되지 않은 집전체 구리 포일 노출 영역입니다. 와인딩 후 젤리롤 상하단으로 노출되어 탭 용접(Tab Welding)이 이루어지는 면적입니다. 무지부 폭이 너무 좁으면 탭 용접 시 활물질 위에 용접되는 불량이 생기고, 너무 넓으면 전극 유효 면적이 줄어 배터리 용량이 감소합니다."},

  # 와인딩 구간
  {"instruction": "와인딩 장비 내부에 번호가 매겨진 롤러들이 많이 보여. 번호가 왜 있는거야?", "input": "감지된 객체: Numbered_Core, Winding_Cell", "output": "번호는 각 롤러의 위치와 역할을 명확히 식별하기 위해 부착됩니다. 와인딩 장비는 수십 개의 롤러로 구성되어 있어 번호 없이는 특정 롤러를 찾기 어렵습니다. 롤러 번호는 장비 도면과 연결되어 유지보수 시 신속한 점검이 가능하며, 문제 발생 시 어느 번호 롤러에서 이상이 생겼는지 즉시 파악할 수 있습니다."},
  {"instruction": "압롤(压辊)이라고 표기된 파란색 장치가 보여. 이게 뭐야?", "input": "감지된 객체: Numbered_Core, Winding_Cell", "output": "압롤(压辊, Pressure Roller)은 와인딩 시 전극과 분리막이 맨드릴에 감길 때 공기가 들어가지 않도록 눌러주는 가압 롤러입니다. 권취 과정에서 발생하는 공기 포켓을 제거하여 젤리롤 내부를 치밀하게 만듭니다. 압력이 너무 강하면 분리막 손상, 너무 약하면 권취 불량이 발생하므로 적정 압력 제어가 중요합니다."},
  {"instruction": "흰색 분리막 롤이 와인딩 장비에 설치되어 있어. 분리막이 전극보다 폭이 넓어 보여. 왜 그래?", "input": "감지된 객체: Separator, Winding_Cell", "output": "분리막이 전극보다 넓게 설계된 것은 오버행(Overhang) 구조 때문입니다. 전극 정렬에 약간의 오차가 생겨도 분리막이 충분히 넓으면 양극과 음극이 직접 접촉하지 않습니다. 일반적으로 분리막은 전극보다 양쪽으로 각 0.5~2mm 더 넓게 설계됩니다. 이 오버행 설계가 내부 단락 방지의 핵심입니다."},
  {"instruction": "와인딩 맨드릴 주변에서 전극이 빠르게 감기고 있어. 초기 권취 단계에서 중요한게 뭐야?", "input": "감지된 객체: Winding_Cell, Separator", "output": "초기 권취 단계에서는 전극과 분리막의 시작 위치 정렬이 가장 중요합니다. 시작 테이프로 분리막을 맨드릴에 고정하고 전극이 정확한 오버행을 유지하며 감기기 시작해야 합니다. 초기 몇 바퀴의 권취 품질이 전체 젤리롤 정렬에 영향을 미칩니다. 자동 비전 정렬 시스템이 초기 단계부터 위치를 모니터링합니다."},
  {"instruction": "와인딩 장비 내부 커터가 전극을 절단하는 모습이 보여. 권취 완료 후 어떻게 마무리해?", "input": "감지된 객체: Winding_Cell, Core_Shaft", "output": "권취 완료 후 자동 커터가 설계된 전극 길이에서 정확하게 절단합니다. 절단 후 전극 끝부분이 젤리롤 외부로 나오지 않도록 분리막으로 한 바퀴 더 감습니다. 마지막으로 종료 테이프를 부착하여 분리막 끝을 고정하여 젤리롤이 풀리지 않게 합니다. 이후 단락 시험으로 내부 불량을 확인합니다."},
  {"instruction": "완성된 젤리롤이 검은 받침대에 놓여있어. 원통형 모양인데 이게 어떤 배터리에 쓰여?", "input": "감지된 객체: Winding_Cell, Core_Shaft", "output": "원통형 젤리롤은 원통형 배터리 셀에 사용됩니다. 대표적으로 18650(지름 18mm, 높이 65mm), 21700(지름 21mm, 높이 70mm), 4680(지름 46mm, 높이 80mm) 규격이 있습니다. 전기차용 배터리팩, 전동공구, ESS 등에 널리 사용됩니다. 최근 테슬라가 채택한 4680 대형 원통형 셀이 주목받고 있습니다."},

  # ===== AI 에이전트 연결 시나리오 =====
  {"instruction": "불량이 감지됐어. 어떤 조치를 취해야 해?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 불량이 감지되었습니다. 즉시 다음 조치를 취하세요. 1단계: 라인 속도를 낮추고 불량 구간을 마킹합니다. 2단계: 슬롯 다이 상태를 점검하여 막힘이나 이물질 여부를 확인합니다. 3단계: 슬러리 점도와 압력을 재확인합니다. 4단계: 불량 구간이 다음 공정으로 투입되지 않도록 격리합니다. 5단계: 정상화 후 재검사를 실시합니다."},
  {"instruction": "불량이 감지됐어. 어떤 조치를 취해야 해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 불량이 감지되었습니다. 즉시 다음 조치를 취하세요. 1단계: 라인을 정지하고 불량 구간을 마킹합니다. 2단계: 슬리팅 블레이드의 마모 상태를 점검합니다. 3단계: 필요시 블레이드를 교체합니다. 4단계: 집진 시스템 작동 상태를 확인합니다. 5단계: 교체 후 절단면 품질을 재검사하고 생산을 재개합니다."},
  {"instruction": "불량이 감지됐어. 어떤 조치를 취해야 해?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 불량이 감지되었습니다. 즉시 다음 조치를 취하세요. 1단계: 라인을 정지하고 현재 권취 중인 젤리롤을 격리합니다. 2단계: 전극과 분리막의 정렬 상태를 점검합니다. 3단계: 장력 제어 시스템을 재설정합니다. 4단계: 분리막 손상 여부를 확인합니다. 5단계: 시험 권취 후 품질 확인 후 생산을 재개합니다."},
  {"instruction": "현재 공정 상태 리포트를 생성해줘.", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 상태 리포트입니다. 탐지된 장비: 슬롯 다이(Slot Die), 전극 포일(Electrode Foil). 현재 공정: 코팅(Coating) - 배터리 전극 제조 1단계. 주요 모니터링 항목: 코팅 두께 균일성, 슬러리 도포 상태, 건조로 온도, 표면 결함(핀홀/줄무늬). 다음 공정: 캘린더링(Calendering). 권장 조치: 인라인 두께 측정값을 확인하고 설정값과 비교하여 편차를 모니터링하세요."},
  {"instruction": "현재 공정 상태 리포트를 생성해줘.", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정 상태 리포트입니다. 탐지된 장비: 슬리팅 블레이드(Slitting Blade), 전극 포일(Electrode Foil). 현재 공정: 슬리팅(Slitting) - 배터리 전극 제조 3단계. 주요 모니터링 항목: 절단면 버 발생 여부, 전극 폭 정확도, 금속 분말 집진 상태, 커터 마모도. 다음 공정: 와인딩(Winding). 권장 조치: 절단면 품질 카메라 영상을 확인하고 커터 교체 주기를 점검하세요."},
  {"instruction": "현재 공정 상태 리포트를 생성해줘.", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정 상태 리포트입니다. 탐지된 장비: 와인더(Winder), 분리막(Separator), 전극 포일(Electrode Foil). 현재 공정: 와인딩(Winding) - 배터리 전극 제조 4단계(최종). 주요 모니터링 항목: 전극 정렬 편차, 권취 장력, 분리막 손상 여부, 젤리롤 직경. 다음 공정: 탭 용접(Tab Welding). 권장 조치: 비전 정렬 시스템 수치를 확인하고 젤리롤 직경이 설계값 범위 내에 있는지 모니터링하세요."},
  {"instruction": "이 공정에서 다음 공정으로 넘어가도 될지 판단해줘.", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정에서 캘린더링 공정으로 넘어가기 위한 체크리스트입니다. 확인 항목: 1) 코팅 두께가 설계값 ±5% 이내인가? 2) 표면 결함(핀홀, 줄무늬)이 없는가? 3) 건조가 완전히 완료되었는가? 4) 무지부 폭이 설계값에 맞는가? 5) 불량 구간이 마킹되었는가? 모든 항목이 확인되면 캘린더링 공정으로 투입 가능합니다. 불량 구간은 캘린더링 후 슬리팅 시 제거합니다."},
  {"instruction": "이 공정에서 다음 공정으로 넘어가도 될지 판단해줘.", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정에서 와인딩 공정으로 넘어가기 위한 체크리스트입니다. 확인 항목: 1) 전극 폭이 설계값 ±0.1mm 이내인가? 2) 절단면에 버(Burr)가 없는가? 3) 롤 단면이 균일하게 감겼는가(텔레스코핑 없음)? 4) 금속 분말 오염이 없는가? 5) 각 가닥 길이가 설계값에 맞는가? 모든 항목이 확인되면 와인딩 공정으로 투입 가능합니다."},
  {"instruction": "작업자에게 경고 메시지를 보내야 해. 뭐라고 해야 해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정 경고 메시지: 현재 슬리팅 블레이드 마모가 감지되었습니다. 절단면 품질을 즉시 육안으로 확인하세요. 버(Burr) 발생이 확인되면 라인을 정지하고 블레이드를 교체해야 합니다. 금속 분말 집진 시스템 작동 여부도 함께 확인해 주세요. 불량 전극이 와인딩 공정으로 투입되지 않도록 주의하세요."},
  {"instruction": "작업자에게 경고 메시지를 보내야 해. 뭐라고 해야 해?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정 경고 메시지: 전극 정렬 편차가 감지되었습니다. 비전 정렬 시스템 수치를 즉시 확인하세요. 편차가 허용 범위를 초과하면 라인을 정지하고 장력 제어 시스템을 재설정해야 합니다. 현재 권취 중인 젤리롤의 분리막 손상 여부도 확인해 주세요. 불량 젤리롤은 즉시 격리하고 품질팀에 보고하세요."},
  {"instruction": "이 공정을 자동화하려면 어떤 AI 기능이 필요해?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 자동화를 위해 필요한 AI 기능은 다음과 같습니다. 1) 실시간 표면 결함 탐지: 딥러닝 기반 비전 검사로 핀홀, 줄무늬, 두께 불균일을 자동 감지합니다. 2) 코팅 두께 예측 모델: 슬러리 점도, 속도, 온도 데이터로 코팅 두께를 예측하고 자동 조정합니다. 3) 이상 탐지: 공정 파라미터 이상을 사전에 감지하여 불량 발생 전 경보를 발령합니다. 4) 자연어 리포트: 공정 상태를 자동으로 분석하고 작업자에게 한국어로 설명합니다."},
  {"instruction": "이 공정을 자동화하려면 어떤 AI 기능이 필요해?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정 자동화를 위해 필요한 AI 기능은 다음과 같습니다. 1) 실시간 정렬 제어: 비전 AI가 전극과 분리막 정렬 편차를 감지하고 자동 보정 신호를 전송합니다. 2) 젤리롤 품질 예측: 권취 장력과 속도 데이터로 젤리롤 품질을 실시간 예측합니다. 3) 이상 탐지: 분리막 손상과 정렬 불량을 사전에 감지합니다. 4) 자연어 공정 설명: 탐지된 객체 정보를 기반으로 현재 공정 상태를 자동으로 분석하여 보고합니다."},
]

vision_qa.extend(additional2)

with open('/workspace/llm_project/data/vision_qa.json', 'w', encoding='utf-8') as f:
    json.dump(vision_qa, f, ensure_ascii=False, indent=2)

print(f"저장 완료! 총 {len(vision_qa)}개")

저장 완료! 총 88개


In [4]:
import json

with open('/workspace/llm_project/data/vision_qa.json', 'r', encoding='utf-8') as f:
    vision_qa = json.load(f)

additional3 = [
  {"instruction": "지금 공정에서 수율을 높이려면 어떻게 해야 해?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 수율 향상을 위해서는 첫째, 슬롯 다이 갭을 정밀하게 조정하여 코팅 두께 편차를 최소화합니다. 둘째, 슬러리 점도를 일정하게 유지하여 도포 균일성을 확보합니다. 셋째, 인라인 비전 검사로 불량을 조기 탐지하여 불량 구간을 최소화합니다. 넷째, 건조로 온도 프로파일을 최적화하여 용매를 완전히 제거합니다."},
  {"instruction": "지금 공정에서 수율을 높이려면 어떻게 해야 해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 공정 수율 향상을 위해서는 첫째, 블레이드 교체 주기를 데이터 기반으로 최적화하여 버 발생을 예방합니다. 둘째, EPC 시스템으로 사행을 자동 보정하여 폭 불량을 줄입니다. 셋째, 집진 시스템을 상시 점검하여 금속 분말 오염을 방지합니다. 넷째, 절단면 인라인 검사로 불량 구간을 즉시 마킹합니다."},
  {"instruction": "지금 공정에서 수율을 높이려면 어떻게 해야 해?", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정 수율 향상을 위해서는 첫째, 비전 정렬 시스템으로 전극 정렬을 실시간 자동 보정합니다. 둘째, 장력 제어를 최적화하여 공기 포켓 발생을 최소화합니다. 셋째, 분리막 공급 장력을 균일하게 유지합니다. 넷째, 젤리롤 완성 후 즉시 단락 시험을 실시하여 불량을 조기에 걸러냅니다."},
  {"instruction": "이 공정 데이터를 MES 시스템에 어떻게 연동해야 해?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 데이터를 MES에 연동하려면 첫째, 인라인 두께 측정값과 결함 탐지 결과를 실시간으로 MES에 전송합니다. 둘째, 코팅 롤 단위로 배치 번호를 부여하여 트레이서빌리티를 확보합니다. 셋째, 불량 구간 마킹 정보를 MES에 기록하여 이후 공정에서 해당 구간을 자동으로 제외합니다. 넷째, 공정 파라미터(속도, 온도, 갭)를 MES에 기록합니다."},
  {"instruction": "예방 정비가 필요한 시점을 어떻게 예측해?", "input": "감지된 객체: Slitting_Blade, Electrode_Foil", "output": "슬리팅 블레이드 예방 정비 시점 예측을 위해서는 첫째, 누적 절단 길이를 모니터링하여 설계 수명에 도달하기 전에 교체 일정을 수립합니다. 둘째, 인라인 절단면 품질 데이터를 분석하여 품질 저하 추세를 감지합니다. 셋째, AI 기반 예측 모델로 블레이드 마모 패턴을 학습하여 교체 시점을 자동 예측합니다. 넷째, 경보 임계값에 도달하면 작업자에게 자동으로 알람을 전송합니다."},
  {"instruction": "이 공정에서 에너지 효율을 높이려면?", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 에너지 효율 향상을 위해서는 첫째, 건조로 온도 프로파일을 최적화하여 과도한 가열을 방지합니다. 둘째, NMP 회수 시스템을 효율적으로 운영하여 용매 손실을 최소화합니다. 셋째, 코팅 속도를 최적화하여 불필요한 재작업을 줄입니다. 넷째, 인버터 모터로 속도 제어를 통해 전력 소비를 줄입니다."},
  {"instruction": "작업자 교육을 위해 지금 공정을 설명해줘.", "input": "감지된 객체: Slot_Die, Electrode_Foil", "output": "코팅 공정 작업자 교육 설명입니다. 현재 화면에 보이는 슬롯 다이(Slot Die)는 배터리 전극에 활물질을 입히는 핵심 장비입니다. 구리색 전극 포일이 슬롯 다이 아래를 지나면서 검은색 활물질 슬러리가 도포됩니다. 코팅 후 대형 건조로를 통과하여 용매를 제거합니다. 주요 점검 사항은 코팅 두께 균일성, 표면 결함, 건조 상태입니다."},
  {"instruction": "작업자 교육을 위해 지금 공정을 설명해줘.", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 공정 작업자 교육 설명입니다. 현재 화면에서 와인더(Winder)가 전극 포일과 흰색 분리막(Separator)을 맨드릴에 감아 원통형 젤리롤을 만들고 있습니다. 분리막은 양극과 음극이 직접 닿지 않도록 가운데서 분리하는 역할을 합니다. 전극과 분리막이 정확히 정렬되어야 하며, 정렬 불량은 배터리 단락과 발화로 이어질 수 있습니다."},
  {"instruction": "품질 이상 원인을 분석해줘.", "input": "감지된 객체: Press_Roller, Electrode_Foil", "output": "캘린더링 공정 품질 이상 원인 분석입니다. 전극 두께 불균일 원인: 롤러 압력 편차, 롤러 표면 이물질, 원재료(코팅 전극) 두께 편차. 전극 크랙 원인: 과도한 압연 압력, 낮은 바인더 함량. 주름 원인: 장력 불균일, 롤러 속도 편차. 활물질 탈락 원인: 바인더 부족 또는 건조 불량. 각 불량 유형에 맞는 공정 파라미터를 점검하고 조정하세요."},
  {"instruction": "이 장비의 유지보수 체크리스트를 알려줘.", "input": "감지된 객체: Winder, Separator, Electrode_Foil", "output": "와인딩 장비 유지보수 체크리스트입니다. 일일 점검: 롤러 표면 청결 상태, 장력 센서 영점 확인, 비전 정렬 시스템 캘리브레이션. 주간 점검: 맨드릴 마모 상태, 커터 날 상태, 분리막 가이드 롤러 마모. 월간 점검: 모터 및 베어링 점검, 장력 제어 시스템 정밀 교정, 비전 카메라 렌즈 청소. 분기 점검: 전체 구동계 오버홀, 소프트웨어 업데이트."},
  {"instruction": "현재 탐지된 객체로 어느 공정인지 판단하고 다음 조치를 알려줘.", "input": "감지된 객체: Separator, Numbered_Core, Winding_Cell", "output": "탐지 결과 분석: 분리막(Separator), 번호 코어(Numbered Core), 와인딩 셀(Winding Cell)이 감지되었습니다. 판단: 와인딩(Winding) 공정 진행 중입니다. 권장 조치: 1) 비전 정렬 시스템으로 분리막과 전극의 정렬 편차를 확인하세요. 2) 번호 코어 롤러의 장력 상태를 점검하세요. 3) 젤리롤 직경이 설계값 범위인지 확인하세요. 4) 이상 없으면 공정을 계속 진행하세요."},
  {"instruction": "현재 탐지된 객체로 어느 공정인지 판단하고 다음 조치를 알려줘.", "input": "감지된 객체: Core_Shaft, Winding_Cell", "output": "탐지 결과 분석: 코어 샤프트(Core Shaft), 와인딩 셀(Winding Cell)이 감지되었습니다. 판단: 와인딩(Winding) 공정의 권취 완료 단계입니다. 권장 조치: 1) 젤리롤 외경이 설계값 범위인지 확인하세요. 2) 종료 테이프가 정상적으로 부착되었는지 확인하세요. 3) 코어 샤프트 제거 후 젤리롤 형태를 점검하세요. 4) 단락 시험을 실시하여 내부 불량 여부를 확인하세요. 5) 합격 젤리롤을 탭 용접 공정으로 이송하세요."},
]

vision_qa.extend(additional3)

with open('/workspace/llm_project/data/vision_qa.json', 'w', encoding='utf-8') as f:
    json.dump(vision_qa, f, ensure_ascii=False, indent=2)

print(f"저장 완료! 총 {len(vision_qa)}개")

저장 완료! 총 100개


In [5]:
train_script = '''
import torch
import json
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "/workspace/llm_project/output_v9"
DATA_PATH = "/workspace/llm_project/data/vision_qa.json"
SYSTEM_PROMPT = "당신은 배터리 제조 공정 전문 AI입니다. 비전 시스템이 감지한 객체 정보를 바탕으로 한국어로만 정확하게 답변하세요."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

def format_text(instruction, input_ctx, output):
    user_content = f"{instruction}\\n\\n현재 상황: {input_ctx}" if input_ctx else instruction
    return (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\\n"
        f"{user_content}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\\n"
        f"{output}<|eot_id|>"
    )

with open(DATA_PATH) as f:
    raw = json.load(f)

dataset = Dataset.from_list([
    {"text": format_text(x["instruction"], x.get("input", ""), x["output"])}
    for x in raw
])
split = dataset.train_test_split(test_size=0.15, seed=42)
print(f"학습: {len(split[\'train\'])}개 / 검증: {len(split[\'test\'])}개")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=15,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="steps",
    save_steps=50,
    lr_scheduler_type="cosine",
    report_to="tensorboard",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=tokenizer,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
print("학습 완료!")
'''

with open('/workspace/llm_project/train_v9.py', 'w') as f:
    f.write(train_script)
print("train_v9.py 저장 완료!")

train_v9.py 저장 완료!


In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
ADAPTER_DIR = "/workspace/llm_project/output_v9"
SYSTEM_PROMPT = "당신은 배터리 제조 공정 전문 AI입니다. 비전 시스템이 감지한 객체 정보를 바탕으로 한국어로만 정확하게 답변하세요."

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

def ask(question, detected_objects=""):
    user_content = f"{question}\n\n현재 상황: {detected_objects}" if detected_objects else question
    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{user_content}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.3,
            do_sample=True,
            repetition_penalty=1.2,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("assistant\n")[-1].strip()

print("Q1:", "지금 여기선 뭘 하고 있는거야?")
print(ask("지금 여기선 뭘 하고 있는거야?", "감지된 객체: Slitting_Blade, Electrode_Foil"))
print("---")
print("Q2:", "불량이 감지됐어. 어떤 조치를 취해야 해?")
print(ask("불량이 감지됐어. 어떤 조치를 취해야 해?", "감지된 객체: Winder, Separator, Electrode_Foil"))

ModuleNotFoundError: No module named 'transformers'